# Tutorial 04 - Exercise 03: Hospital Indexing & Selection

Demonstrates custom label indexing, `.loc` vs `.iloc` selection, boolean
filtering, and department subsetting.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

Locates the `APDS/06_Hospital_Department_Analytics/` project folder (anchored on `data/hospital_original.csv`) by walking upward from the
current working directory — so the notebook runs identically from the repository
root, the project folder or its `notebooks/` sub-folder. On Google Colab the
repository is cloned into `/content/APSD_TAI_LAB` automatically (opening the
notebook directly from GitHub also works). The resolved project folder is added
to `sys.path` so its local modules (e.g. `utils`, `students`) import cleanly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "06_Hospital_Department_Analytics"
ANCHOR_REL = "data/hospital_original.csv"  # file that uniquely identifies the project folder
REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "APDS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
    PROJ_DIR = _repo / "APDS" / PROJECT_NAME
else:
    # Local IDE / Jupyter: find the project folder by walking upward from the
    # working directory (works from the repo root, the project folder or its
    # notebooks/ sub-folder, regardless of where Jupyter was launched).
    _cwd = Path.cwd().resolve()
    PROJ_DIR = next(
        (
            c / "APDS" / PROJECT_NAME
            for c in (_cwd, *_cwd.parents)
            if (c / "APDS" / PROJECT_NAME / ANCHOR_REL).exists()
        ),
        None,
    )
    if PROJ_DIR is None:
        raise FileNotFoundError(
            f"Project folder APDS/{PROJECT_NAME} (anchor: {ANCHOR_REL}) not found "
            f"above {_cwd}. Launch Jupyter from inside the cloned repository."
        )
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print(f"Project folder : {PROJ_DIR}")
print(f"Google Colab   : {IN_COLAB}")


### 1. Load and Label the DataFrame


In [ ]:
import pandas as pd

df = pd.read_csv(PROJ_DIR / "data" / "hospital_original.csv")
df.index = ["A", "B", "C", "D", "E", "F", "G", "H"]
display(df)


### 2. Label vs Position Slicing


In [ ]:
print("Slice labels C to F:")
display(df.loc["C":"F"])

print("Select positions 1, 2, and 3:")
display(df.iloc[1:4])

print("Columns B to F (City, Department, DailyCost):")
display(df.loc["B":"F", ["City", "Department", "DailyCost"]])


### 3. High Risk Filtering


In [ ]:
older_dissatisfied = df[(df["Age"] > 45) & (df["Satisfaction"] < 4.0)]
display(older_dissatisfied[["City", "Age", "Satisfaction", "Readmitted"]])


### 4. Neurology Department Focus


In [ ]:
df_neuro = df[df["Department"] == "Neurology"][["City", "Age", "DaysAdmitted", "DailyCost"]].reset_index(drop=True)
df_neuro = df_neuro.set_index("City").sort_values(by="DailyCost", ascending=False)
display(df_neuro)

print("\nNeurology Patient Counts by City:")
print(df_neuro.index.value_counts())
